# Case 3 — Prediction threshold: calibration dynamics

**Reproduces:** Fig 4.7, 4.10 (early calibration instability), Fig 4.11 (mid-stream recall collapse)

Group anomalies, window mode. Transformer-VAE at a low threshold (0.3) is expected to over-fire early in the stream before its rolling threshold catches up (compare against 0.6 baseline and 0.9). LSTM-VAE at a high threshold (0.9) is expected to show the opposite failure mode: precision stays high but recall collapses once the anomaly scores fall below the (too high) decision boundary.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the small synthetic ERA5-shaped dataset shipped with the repo (`scripts/generate_mini_era5.py`) — no data download, no license issues. Numbers will differ from the thesis's real-ERA5 figures (much smaller warmup/test period, noisier), but the *qualitative* effect described above should still show up.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# TODO: update this URL once the repo is pushed to GitHub
REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Generates data/era5/*.csv — fully synthetic, ERA5-shaped, no download needed
!python scripts/generate_mini_era5.py

## Run the suite

`notebooks/cases/case03_threshold_calibration_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially on the one Colab GPU; each is small (mini dataset), so the whole case should finish in a few minutes.

In [ ]:
!python run_regression.py notebooks/cases/case03_threshold_calibration_suite.yaml \
    --session runs/regression/case03_threshold_calibration --gpus 0

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case03_threshold_calibration

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case03_threshold_calibration/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Custom view: F1 over equal-length stream segments (not produced by cross_compare.py).
# This approximates the thesis's 'anomaly-equal bucket' plots (Fig 4.6/4.10/4.11/4.22):
# chop each run's trial_predictions.csv into N equal-length chunks and compute F1 per chunk.
import glob
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

SESSION = "runs/regression/case03_threshold_calibration"
N_BUCKETS = 8

# leaf dir name (hostname_seed_N) varies by machine, so glob it rather than hardcode it
run_globs = {
    "TF threshold=0.3": f"{SESSION}/case03_threshold_calibration/case03_threshold_calibration_suite/Group/TF_VAE/threshold_0_3/*_seed_42/trial_predictions.csv",
    "TF threshold=0.6 (baseline)": f"{SESSION}/case03_threshold_calibration/case03_threshold_calibration_suite/Group/TF_VAE/threshold_0_6/*_seed_42/trial_predictions.csv",
    "TF threshold=0.9": f"{SESSION}/case03_threshold_calibration/case03_threshold_calibration_suite/Group/TF_VAE/threshold_0_9/*_seed_42/trial_predictions.csv",
    "LSTM threshold=0.9": f"{SESSION}/case03_threshold_calibration/case03_threshold_calibration_suite/Group/LSTM/threshold_0_9/*_seed_42/trial_predictions.csv",
}
runs = {label: sorted(glob.glob(pattern))[0] for label, pattern in run_globs.items()}

def bucket_f1(csv_path, n_buckets):
    df = pd.read_csv(csv_path, skiprows=1, header=None,
                      names=["ts_start","ts_end","step","confusion","label",
                             "final_pred","trained","final_conf","metrics"])
    df["bucket"] = pd.cut(df["step"], n_buckets, labels=False)
    out = [float("nan")] * n_buckets   # fixed length — a bucket with no rows stays NaN (gap in the line)
    for b, g in df.groupby("bucket"):
        tp = (g["confusion"] == "TP").sum(); fp = (g["confusion"] == "FP").sum()
        fn = (g["confusion"] == "FN").sum()
        prec = tp / (tp + fp) if (tp + fp) else float("nan")
        rec  = tp / (tp + fn) if (tp + fn) else float("nan")
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) and prec == prec and rec == rec and (prec + rec) > 0 else float("nan")
        out[int(b)] = f1
    return out

fig, ax = plt.subplots(figsize=(9, 4))
for label, path in runs.items():
    f1s = bucket_f1(path, N_BUCKETS)
    ax.plot(range(1, N_BUCKETS + 1), f1s, marker="o", label=label)
ax.set_xlabel("Stream segment (1..8, equal length)")
ax.set_ylabel("F1 (within segment)")
ax.set_title("F1 across the stream — low threshold: early instability; high threshold: recall collapse")
ax.legend()
ax.set_ylim(-0.05, 1.05)
plt.show()